# Task 3 — Gender occasional grayscale (10%)

Run All trains **folds 0 and 4 only**, from scratch. Keep **dropout 0.30 and the completed 04aa mild-darkening policy**, then make each training image grayscale with probability **0.10**. About 90% of image presentations retain color; the exact realized fraction is random. This tests whether occasional loss of color helps the model also use shape and texture.

Use a **fresh Colab L4** runtime matching 04w/04aa. Push this notebook and its source changes before running. G2/E6 runs, original dropout runs, completed 04aa runs and 04w precision evidence must be on Drive. The 0.45 dropout runs are not parents of this trial.


## 1. Colab GPU and repository


In [1]:
import os
import shutil
import subprocess
import sys
import time
import zipfile
from pathlib import Path

REPO_URL = "https://github.com/TrnLin/MLA2.git"
BRANCH = "fashion-analysis-and-cleanup"
CHECKOUT_DIR = Path("/content/MLA2")
DRIVE_MOUNT = Path("/content/drive")
DRIVE_PROJECT_DIR = DRIVE_MOUNT / "MyDrive/MLA2"
DATA_ZIP = DRIVE_PROJECT_DIR / "data/task3-data.zip"
LOCAL_DATA_ZIP = Path("/content/task3-data.zip")
DRIVE_TASK_DIR = DRIVE_PROJECT_DIR / "task3"
DRIVE_REGISTRY = DRIVE_TASK_DIR / "results/runs.csv"

def run_checked(command, *, cwd=None):
    command = [str(part) for part in command]
    print("$", " ".join(command), flush=True)
    return subprocess.run(command, cwd=cwd, check=True)

try:
    from google.colab import drive
except ImportError as exc:
    raise RuntimeError("Connect this notebook to a Google Colab GPU runtime first.") from exc

drive.mount(str(DRIVE_MOUNT), force_remount=False)
if (CHECKOUT_DIR / ".git").is_dir():
    remote_url = subprocess.check_output(
        ["git", "remote", "get-url", "origin"], cwd=CHECKOUT_DIR, text=True
    ).strip()
    if remote_url != REPO_URL:
        raise RuntimeError(f"{CHECKOUT_DIR} belongs to a different repository: {remote_url}")
    run_checked(["git", "fetch", "origin", BRANCH], cwd=CHECKOUT_DIR)
    run_checked(["git", "switch", BRANCH], cwd=CHECKOUT_DIR)
    run_checked(["git", "merge", "--ff-only", f"origin/{BRANCH}"], cwd=CHECKOUT_DIR)
elif CHECKOUT_DIR.exists():
    raise RuntimeError(f"{CHECKOUT_DIR} exists but is not a Git repository.")
else:
    run_checked(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, CHECKOUT_DIR])

commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=CHECKOUT_DIR, text=True).strip()
REPO_DIR = CHECKOUT_DIR / "core" if (CHECKOUT_DIR / "core/src/fashion").is_dir() else CHECKOUT_DIR
LOCAL_REGISTRY = REPO_DIR / "results/runs.csv"
print("Repository ready:", REPO_DIR)
print("Commit:", commit)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
$ git fetch origin task-3-gender-usage-classification
$ git switch task-3-gender-usage-classification
$ git merge --ff-only origin/task-3-gender-usage-classification
Repository ready: /content/MLA2
Commit: a944853bb441c2d8cf8a4fd410b800344cf42001


## 2. Teacher data and canonical split


In [2]:
def copy_teacher_zip_to_local_disk():
    if LOCAL_DATA_ZIP.is_file():
        try:
            with zipfile.ZipFile(LOCAL_DATA_ZIP) as existing:
                existing.infolist()
            return
        except zipfile.BadZipFile:
            LOCAL_DATA_ZIP.unlink()
    partial = LOCAL_DATA_ZIP.with_suffix(".zip.partial")
    for attempt in range(1, 4):
        partial.unlink(missing_ok=True)
        try:
            expected_bytes = DATA_ZIP.stat().st_size
            with DATA_ZIP.open("rb") as source, partial.open("wb") as target:
                shutil.copyfileobj(source, target, length=8 * 1024**2)
            if partial.stat().st_size != expected_bytes:
                raise OSError("The local ZIP copy is incomplete.")
            partial.replace(LOCAL_DATA_ZIP)
            return
        except OSError as error:
            partial.unlink(missing_ok=True)
            if attempt == 3:
                raise RuntimeError("Drive disconnected three times. Remount and retry.") from error
            drive.mount(str(DRIVE_MOUNT), force_remount=True)
            time.sleep(2)

copy_teacher_zip_to_local_disk()
teacher_dir = REPO_DIR / "data/raw/teacher"
required_files = (
    teacher_dir / "train/styles_train.csv",
    teacher_dir / "test/styles_prediction.csv",
)
image_dirs = (teacher_dir / "train/images_train", teacher_dir / "test/images_test")
image_suffixes = {".jpg", ".jpeg"}
with zipfile.ZipFile(LOCAL_DATA_ZIP) as archive:
    names = archive.namelist()
    if any(Path(name).is_absolute() or ".." in Path(name).parts for name in names):
        raise RuntimeError("The teacher archive contains an unsafe path.")
    expected_images = sum(
        name.startswith("data/raw/teacher/") and Path(name).suffix.lower() in image_suffixes
        for name in names
    )
    current_images = sum(
        path.suffix.lower() in image_suffixes for folder in image_dirs for path in folder.glob("*")
    )
    if current_images != expected_images or not all(path.is_file() for path in required_files):
        archive.extractall(REPO_DIR)

actual_images = sum(
    path.suffix.lower() in image_suffixes for folder in image_dirs for path in folder.glob("*")
)
if actual_images != expected_images or not all(path.is_file() for path in required_files):
    raise RuntimeError(f"Teacher data is incomplete: {actual_images:,}/{expected_images:,} images")

os.chdir(REPO_DIR)
os.environ["FASHION_PROJECT_ROOT"] = str(REPO_DIR)
if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))
(DRIVE_TASK_DIR / "results").mkdir(parents=True, exist_ok=True)
print(f"Teacher data ready: {actual_images:,} images")


Teacher data ready: 44,441 images


## 3. One frozen change: occasional grayscale

The direct parents are **04aa: dropout 0.30 plus mild darkening** on folds 0 and 4. Error inspection found that grayscale reduced correct Girls predictions from 125 to 60 out of 216; increasing dropout to 0.45 added clean-image failures. Start from the 0.30 recipe and change only training augmentation.

**Why 10%?** It is one cautious, predeclared trial, not an optimized rate. Color contains useful information, so most training views keep it. A larger fraction might help more or could remove too much useful information. No sweep or automatic rate increase runs here.

Apply translation ±2 px per axis with probability 0.50, then the existing darkening with probability 0.25 and brightness factor uniformly drawn from 0.90 to 1.00. After these, apply **RGB → L → RGB with probability 0.10**, before normalization. The output remains three-channel RGB. A separate persistent per-worker grayscale random stream leaves the translation and darkening streams unchanged.

Keep widths `[32, 64, 128, 256]`, **390,181 parameters**, full images, fixed GeM p=3, classifier dropout 0.30, clean fold-training RGB normalization, plain cross-entropy, AdamW weight decay 0.0001, the G2 learning rate and cosine schedule, batch 128, 30 epochs, seed 2753 and the final-epoch checkpoint. Every child starts with random weights.

Clean training-score evaluation and ordinary validation use original-color inputs, with training augmentation and dropout disabled. The separate robustness suite still evaluates its fixed grayscale and other corruptions. Catalog labels, canonical splits and evaluation references stay fixed; flagged label–name conflicts are not corrected by this experiment.

The next cell verifies the two exact 04aa parents, their source audit, the original dropout-to-G2-to-E6 lineage and precision evidence. These are research parents, not accepted models. All existing pass rules remain below.


In [3]:
from fashion.train.task3_gender_grayscale import (
    check_gender_grayscale_sources,
    run_gender_grayscale_screen,
)

G2_DIR = DRIVE_TASK_DIR / "experiments/t3_gender_v2_g2_translation/gender"
E6_DIR = DRIVE_TASK_DIR / "experiments/t3_gender_e6_gem_p3/gender"
DROPOUT_DIR = DRIVE_TASK_DIR / "experiments/t3_gender_dropout_030/gender"
DARKENING_DIR = DRIVE_TASK_DIR / "experiments/t3_gender_dropout_030_mild_darkening/gender"
PRECISION_DIR = DRIVE_TASK_DIR / "diagnostics/gender_precision/20260905T085822668071Z"

sources, classes, spec, evidence = check_gender_grayscale_sources(
    g2_directory=G2_DIR, e6_directory=E6_DIR, dropout_directory=DROPOUT_DIR,
    darkening_directory=DARKENING_DIR, source_registry_path=DRIVE_REGISTRY,
    precision_directory=PRECISION_DIR, root=REPO_DIR,
)
assert spec.classifier_dropout == 0.30
assert spec.to_dict()["grayscale_probability"] == 0.10
print("Verified source runs:", {name: len(runs) for name, runs in sources.items()})
print("Direct 0.30 + darkening parents:",
      {fold: run["run_id"] for fold, run in sources["Drop30Dark"].items()})
print("Frozen recipe:", spec.to_dict())
print("Output:", DRIVE_TASK_DIR / spec.artifact_dir / "gender")

Verified source runs: {'G2': 5, 'E6': 5, 'Drop30': 2, 'Drop30Dark': 2}
Direct 0.30 + darkening parents: {0: 't3_gender_dropout_030_mild_darkening_gender_smallcnngem3_f0_s2753_cfbed3f0fed4_20260905T123721Z5d546f', 4: 't3_gender_dropout_030_mild_darkening_gender_smallcnngem3_f4_s2753_cfbed3f0fed4_20260905T124728Z7dfb12'}
Frozen recipe: {'name': 'gender_dropout_030_mild_darkening_grayscale_010', 'target': 'gender', 'experiment_id': 't3_gender_dropout_030_mild_darkening_grayscale_010', 'hypothesis_id': 't3_gender_dropout_030_mild_darkening_grayscale_010', 'artifact_dir': 'experiments/t3_gender_dropout_030_mild_darkening_grayscale_010', 'run_prefix': 't3_gender_dropout_030_mild_darkening_grayscale_010', 'changed_factor': 'occasional_grayscale_p010_after_mild_darkening', 'parent_artifact_dir': 'experiments/t3_gender_dropout_030_mild_darkening', 'parent_run_ids': ['t3_gender_dropout_030_mild_darkening_gender_smallcnngem3_f0_s2753_cfbed3f0fed4_20260905T123721Z5d546f', 't3_gender_dropout_030_mi

## 4. Agreed screen rules

All rules must pass, using the same full-FP32 evaluation for each model:

- Pooled validation macro-F1 falls by at most **0.030 versus matched G2**. The paired whole-family bootstrap 95% lower bound for the difference must be **at least −0.030**. Neither fold may lose more than 0.030. Use 10,000 draws within folds, seed 2753.
- The mean clean training–validation F1 gap falls by at least **0.050**. **Both folds' gaps must shrink.** Use clean evaluation-mode training scores from each finished checkpoint, not online augmented training scores.
- Preserve the existing stricter class guard: no pooled class loses more than **0.020 F1**. Thus the overall 0.030 allowance does not override a class failure. NLL may rise by at most 0.020 and ECE by at most 0.010 versus G2; these measure the quality of model confidence.
- Preserve corruption guards versus matched E6: the translation-induced F1 change improves by at least 0.030; every other standard corruption, including darkening, worsens by at most 0.020. Each corrupted score is measured relative to that model's clean score.
- Exactly **390,181 parameters** and peak allocated GPU memory **strictly below 3,000,000,000 bytes**. Training time and latency are reported without speed caps. A memory failure stops before another fold begins.

The thresholds are derived from the new matched IEEE reference scores; do not substitute the rounded historical values. A pass means this screen met the chosen trade-off, not that the model is accepted or more accurate.


In [4]:
result = run_gender_grayscale_screen(
    g2_directory=G2_DIR, e6_directory=E6_DIR, dropout_directory=DROPOUT_DIR,
    darkening_directory=DARKENING_DIR, source_registry_path=DRIVE_REGISTRY,
    precision_directory=PRECISION_DIR, output_root=DRIVE_TASK_DIR,
    registry_path=DRIVE_REGISTRY, registry_mirrors=(LOCAL_REGISTRY,), root=REPO_DIR,
)
print("Screen:", result["status"])
for row in result.get("folds", []):
    print("Fold", row["fold"], "train F1:", row["candidate_train_f1"],
          "validation F1:", row["candidate_validation_f1"], "gap:", row["candidate_gap"])
for gate in result.get("checks", []):
    if gate["status"] != "pass":
        print(gate)
if "reason" in result:
    print(result["reason"])
if "incremental_comparison" in result:
    incremental = result["incremental_comparison"]
    print("Direct comparison:", result["direct_parent_comparison"])
    print("Validation F1 change versus 0.30 + darkening:", incremental["validation_delta"])
    print("Paired 95% interval:", incremental["validation_interval"])
    for row in incremental["folds"]:
        print("Fold", row["fold"], "gap reduction versus 0.30 + darkening:", row["gap_reduction"])
    print("Class F1 changes versus 0.30 + darkening:", incremental["class_f1_delta"])
    print("Induced corruption changes:", incremental["mean_induced_change_delta"])

IEEE evaluation: t3_gender_v2_g2_translation_gender_smallcnngem3_f0_s2753_cb072542dbdc_20260904T135102Z6698e6
IEEE evaluation: t3_gender_v2_g2_translation_gender_smallcnngem3_f4_s2753_cb072542dbdc_20260904T140011Z2211b1
IEEE evaluation: t3_gender_e6_gem_p3_gender_smallcnngem3_f0_s2753_a8c09286451b_20260831T090059Z0bab1f
IEEE evaluation: t3_gender_e6_gem_p3_gender_smallcnngem3_f4_s2753_a8c09286451b_20260831T093553Z63b5fd
IEEE evaluation: t3_gender_dropout_030_mild_darkening_gender_smallcnngem3_f0_s2753_cfbed3f0fed4_20260905T123721Z5d546f
IEEE evaluation: t3_gender_dropout_030_mild_darkening_gender_smallcnngem3_f4_s2753_cfbed3f0fed4_20260905T124728Z7dfb12
[task3] preparing target=gender fold=0: train=26,220 (before selection=26,220), validation=6,553
[task3] fitting fold-training RGB statistics for target=gender fold=0
[task3] RGB statistics ready for target=gender fold=0
[task3] registered t3_gender_dropout_030_mild_darkening_grayscale_010_gender_smallcnngem3_f0_s2753_eb37119b7e68_20260

## 5. Stop and review

Do not auto-run folds 1–3, refit or open the held-out test. Each fit uses the registry-aware trainer. Complete matching runs are reused only after source-audit, configuration, lineage and artifact checks.

Results are saved under `MyDrive/MLA2/task3/experiments/t3_gender_dropout_030_mild_darkening_grayscale_010/gender`:

- `screen_decision.json`: all 19 unchanged checks versus G2/E6, plus the direct-parent names and run IDs.
- `clean_gap_comparison.csv` and `ieee_oof_predictions.csv`: clean scores and validation probabilities.
- `incremental_comparison.json` and `dropout_corruption_comparison.csv`: clean scores, class F1, paired validation interval, raw corrupted-image F1 and induced changes versus **0.30 + mild darkening, without training grayscale**. The `dropout_*` fields refer to those direct parents.
- `source_audit.json` and `comparison_ieee_v2/`: source hashes and matched IEEE FP32 evaluation, batch 128.

Inspect Boys/Girls clean scores and grayscale scores together. A smaller corrupted-versus-clean drop can come from a worse clean score. A lower training score alone is not success. This trial may fail the clean gap, class or robustness rules. The incremental comparison adds no new gates, and two reused development folds are not independent final-test evidence.
